## COSMOS Lab 7: Linear Regression Big Lab

**UCLA COSMOS Cluster 10: IoT + Data Science**

This notebook is part of the COSMOS data science lab sequence. Work through the code cells in order and discuss the written prompts with your group.


## Introduction

Welcome to **UCLA COSMOS Cluster 10: IoT + Data Science!** As we're planning to move through topics aggressively in this course, to start out, we'll look to do an end-to-end walkthrough of a datascience project, and then ask you to replicate the code yourself for a new dataset.

**Please note: We don't expect you to fully grasp everything happening here in either code or theory. This content will be reviewed throughout the quarter. Rather we hope that by giving you the full perspective on a data science project it will better help to contextualize the pieces as they're covered in class**

In that spirit, we will first work through an example project from end to end to give you a feel for the steps involved.

Here are the main steps:

1. Get the data
2. Visualize the data for insights
3. Preprocess the data for your machine learning algorithm
4. Select a machine learning model and train it
5. Evaluate its performance



## Working with Real Data

It is best to experiment with real-data as opposed to aritifical datasets.

There are many different open datasets depending on the type of problems you might be interested in!

Here are a few data repositories you could check out:
- [UCI Datasets](http://archive.ics.uci.edu/ml/)
- [Kaggle Datasets](https://www.kaggle.com/datasets?fileType=csv)
- [AWS Datasets](https://registry.opendata.aws)

Below we will run through an California Housing example collected from the 1990's.

## Setup

In [ ]:
import sys
assert sys.version_info >= (3, 5) # python>=3.5
import sklearn
#assert sklearn.__version__ >= "0.20" # sklearn >= 0.20

import numpy as np #numerical package in python
%matplotlib inline
import matplotlib.pyplot as plt #plotting package

# to make this notebook's output identical at every run
np.random.seed(42)

#matplotlib magic for inline figures
%matplotlib inline
import matplotlib.pyplot as plt

# Applying the end-to-end ML steps to a different dataset

Ok now it's time to get to work! We will apply what we've learnt to another dataset (airbnb dataset). For this project we will attempt to **predict the airbnb rental price based on other features in our given dataset.**

# Visualizing Data


<!-- BEGIN QUESTION -->
### Load the data + statistics

Let's do the following set of tasks to get us warmed up:
- load the dataset
- display the first few rows of the data
- drop the following columns: name, host_id, host_name, last_review, neighbourhood
- display a summary of the statistics of the loaded data

In [ ]:
import pandas as pd
airbnb =  pd.read_csv('AB_NYC_2019.csv') # we load the pandas dataframe

In [ ]:
airbnb_drop = airbnb.drop(columns=['name', 'host_id', 'host_name', 'last_review', 'neighbourhood']) # drop unneeded columns

In [ ]:
airbnb_drop.describe()

In [ ]:
airbnb_drop.info()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
### Some Basic Visualizations

Let's try another popular python graphics library: Plotly.

You can find documentation and all the examples you'll need here: [Plotly Documentation](https://plotly.com/python/basic-charts/)


Let's start out by getting a better feel for the distribution of rentals in the market.

#### Generate a pie chart showing the distribution of room type (`room_type` in the dataset) across NYC's 'Manhattan' boroughs (filtered by `neighbourhood_group` in the dataset) 

In [ ]:
import plotly.express as px

# filter the dataframe to only in Manhattan
stanten_values = airbnb_drop[airbnb_drop['neighbourhood_group'] == 'Staten Island']
# group by room_type and count the number of occurences
room_types_value_counts = stanten_values['room_type'].value_counts().reset_index()
fig = px.pie(room_types_value_counts, values='count', names='room_type', title='Distribution of Room Type in Staten Island')
fig.show()
# print("Save it to pie_chart.png")

#### Plot the total number_of_reviews per room_type

We now want to see the total number of reviews left for each room type group in the form of a Bar Chart (where the X-axis is the room type group and the Y-axis is a count of review.

This is a two step process:
1.   You'll have to sum up the reviews per room type group **(hint! try using the groupby function)**
2.   Then use Plotly to generate the graph





In [ ]:
room = airbnb_drop.groupby('room_type').sum()
room.head()

In [ ]:
fig = px.bar(room, y='number_of_reviews')
fig.show()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
### Plot a map of airbnbs throughout New York (if it gets too crowded take a subset of the data, and try to make it look nice if you can :) ).

For reference you can use the Matplotlib code above to replicate this graph here.

In [ ]:
airbnb.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)

In [ ]:
miniairbnb = airbnb[airbnb['price'] < 500]
miniairbnb.head()

In [ ]:
# A more interesting plot is to color code (heatmap) the dots
# based on income. The code below achieves this

# load an image of New York
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
nyc_img=mpimg.imread('nyc.png', -1)

# overlay the NYC map on the plotted scatter plot
# note: plt.imshow still refers to the most recent figure
# that hasn't been plotted yet.

# setting up heatmap colors based on median_house_value feature

ax = miniairbnb.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
                 c="price", cmap=plt.get_cmap("jet"),
                    colorbar=False, alpha=0.1, vmax=miniairbnb["price"].quantile(0.9)
)

min_longitude = miniairbnb["longitude"].min()
max_longitude = miniairbnb["longitude"].max()
min_latitude = miniairbnb["latitude"].min()
max_latitude = miniairbnb["latitude"].max()
print(min_longitude, max_longitude, min_latitude, max_latitude)
plt.imshow(nyc_img, extent=[-74.25, -73.70, 40.50, 40.90], alpha=0.5,)

prices = miniairbnb["price"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
norm = plt.Normalize(vmin=prices.min(), vmax=prices.max())
sm = plt.cm.ScalarMappable(cmap=plt.get_cmap("jet"), norm=norm)

cb = plt.colorbar(sm, ax=ax)
cb.ax.set_yticklabels(["$%d"%(round(v)) for v in tick_values], fontsize=14)
cb.set_label('Price', fontsize=16)

cb = plt.colorbar(sm, ax=ax)
cb.set_ticks(tick_values)
cb.formatter = plt.FuncFormatter(lambda v, pos: f"${v:,.0f}")
cb.update_ticks()

plt.show()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
Now try to recreate this plot using Plotly's Scatterplot functionality. Note that the increased interactivity of the plot allows for some very cool functionality

In [ ]:
fig = px.scatter(miniairbnb, x="longitude", y="latitude", color="price", width=800, height=500)

import base64
#set a local image as a background
image_filename = 'nyc.png'
plotly_logo = base64.b64encode(open(image_filename, 'rb').read())  # create bytes, What Python sees: b'iVBORw0KGgo...'

# 
# fig.update_layout(
#     images=[dict(
#         source="data:image/png;base64,{}".format(plotly_logo.decode()),
#         xref="x",
#         yref="y",
#         x=min_longitude,
#         y=max_latitude,
#         sizex=max_longitude-min_longitude,
#         sizey=max_latitude-min_latitude,
#         sizing="stretch",
#         opacity=0.5,
#         layer="below")])

fig.add_layout_image(
    source=f"data:image/png;base64,{plotly_logo.decode()}",
    xref="x",
    yref="y",
    x=min_longitude,
    y=max_latitude,
    sizex=max_longitude-min_longitude,
    sizey=max_latitude-min_latitude,
    sizing="stretch",
    opacity=0.5,
    layer="below"
)

fig.show()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
### Use Plotly to plot the average price of room types in Brooklyn who have at least 10 Reviews.

Like with the previous example you'll have to do a little bit of data engineering before you actually generate the plot.

Generally I'd recommend the following series of steps:
1. Filter the data by neighborhood group and number of reviews to arrive at the subset of data relevant to this graph.
2. Groupby the room type
3. Take the mean of the price for each roomtype group
4. FINALLY (seriously!?!?) plot the result

In [ ]:
subgroup = airbnb[airbnb['neighbourhood_group'] == 'Brooklyn']
subgroup = subgroup[subgroup['number_of_reviews'] >= 10]

In [ ]:
subgroup

In [ ]:
grouped_airbnb = subgroup.groupby('room_type')
mean_prices = grouped_airbnb['price'].mean().reset_index()

fig = px.bar(mean_prices, x='room_type', y='price', title='Average Price by Room Type in Brooklyn (min 10 reviews)')
fig.show()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
# Prepare the Data

In [ ]:
airbnb_drop.head()

### Feature Engineering

Let's create a new binned feature, `price_cat` that will divide our dataset into quintiles (1-5) in terms of price level (you can choose the levels to assign)

Do a value count to check the distribution of values

In [ ]:
# assign each bin a categorical value [1, 2, 3, 4, 5] in this case.
airbnb_drop["price_cat"] = pd.qcut(airbnb_drop["price"], q=5, labels=[1, 2, 3, 4, 5])
# airbnb_drop["price_cat"] = pd.cut(airbnb_drop["price"],
#                                bins=5,
#                                labels=[1, 2, 3, 4, 5])

airbnb_drop["price_cat"].value_counts()

# # whether there is a NaN value in the `price` column
# airbnb_drop['price'].isna().any()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
### Data Imputation

Determine if there are any null-values and impute them.


In [ ]:
airbnb_drop.info()

In [ ]:
median = airbnb_drop["number_of_reviews"].median()
airbnb_drop["reviews_per_month"].fillna(median, inplace=True)
airbnb_drop.info()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
### Numeric Conversions

Finally, review what features in your dataset are non-numeric and convert them.

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

airbnb_drop['neighbourhood_group'] = labelencoder.fit_transform(airbnb_drop['neighbourhood_group'])
airbnb_drop['room_type'] = labelencoder.fit_transform(airbnb_drop['room_type'])
# airbnb_drop['price_cat'] = labelencoder.fit_transform(airbnb_drop['price_cat'])

airbnb_drop.info()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
# Prepare Data for Machine Learning

Using our `StratifiedShuffleSplit` function example from above, let's split our data into a 80/20 Training/Testing split using `price_cat` to partition the dataset

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
# let's first start by creating our train and test sets

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(airbnb_drop, airbnb_drop["price_cat"]):
    train_set = airbnb_drop.loc[train_index]
    test_set = airbnb_drop.loc[test_index]

In [ ]:
test_set.info()

Finally, remove your labels `price` and `price_cat` from your testing and training cohorts, and create separate label features.

In [ ]:
training = train_set.drop(["price", "price_cat"], axis=1) # drop labels for training set features
train_labels = train_set["price"].copy()

testing = test_set.drop(["price", "price_cat"], axis=1) # drop labels for training set features
test_labels = test_set["price"].copy()

In [ ]:
training.head()

<!-- END QUESTION -->
<!-- BEGIN QUESTION -->
# Fit a linear regression model

The task is to predict the price, you could refer to the housing example on how to train and evaluate your model using **MSE**.
Provide both **test and train set MSE values**.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(training, train_labels)
train_pred = lin_reg.predict(training)

from sklearn.metrics import mean_squared_error
train_mse = mean_squared_error(train_labels, train_pred)
test_mse = mean_squared_error(test_labels, lin_reg.predict(testing))
print("Train MSE", train_mse)
print("Test MSE", test_mse)

<!-- END QUESTION -->